In [ ]:
import pandas as pd
import numpy as np
from joblib import load

from spring_temp_model import predict_spring_temp
from transparency_model import predict_am_transparency, predict_pm_transparency
from fish_survival_model import predict_fish_survival, get_survival_pipeline
from fish_survival_model import prepare_fish_data


🚀 Evaluating model for: Spring Temp (F)
Evaluating XGBoost Regressor...
Fitting 5 folds for each of 18 candidates, totalling 90 fits


In [ ]:
# === Example Input Template ===
def get_base_input():
    print("\n📥 Please enter the following values:")
    month = int(input("Month (1-12): "))
    day = int(input("Day (1-31): "))
    year = int(input("Year (e.g. 2024): "))
    max_temp = float(input("Max air temp: "))
    min_temp = float(input("Min air temp: "))
    dec_rain = float(input("December rain: "))
    calmar_rain = float(input("Calmar rain: "))
    fish_count = int(input("Number of fish: "))
    year_class = int(input("Year class: "))

    df = pd.DataFrame([{
        "Month": month,
        "Day": day,
        "Year": year,
        "Max air temp": max_temp,
        "Min air temp": min_temp,
        "Dec Rain": dec_rain,
        "Calmar Rain": calmar_rain,
        "# fish": fish_count,
        "Year class": year_class
    }])

    # Derived Features
    df["Total Rain"] = df["Dec Rain"] + df["Calmar Rain"]
    df["Spring_Temp x Rain"] = np.nan  # to be filled
    df["Max Air Temp x Rain"] = df["Max air temp"] * df["Total Rain"]
    df["Day of Year"] = pd.to_datetime({"year": df["Year"], "month": df["Month"], "day": df["Day"]}).dt.dayofyear
    df["Fish Age"] = df["Year"] - df["Year class"]
    df["Season"] = df["Month"].apply(lambda m: "Winter" if m in [12,1,2] else "Spring" if m in [3,4,5] else "Summer" if m in [6,7,8] else "Fall")

    return df

# === Full Chained Prediction ===
def chained_prediction():
    base_input = get_base_input()

    # Predict Spring Temp
    spring_temp = predict_spring_temp(base_input)
    print(f"\n🌡️ Predicted Spring Temp (F): {spring_temp:.2f}")
    base_input["Spring Temp (F)"] = spring_temp
    base_input["Spring_Temp x Rain"] = spring_temp * base_input["Total Rain"]

    # Predict AM Transparency
    am_transparency = predict_am_transparency(base_input)
    print(f"\n☀️ Predicted AM Transparency: {am_transparency:.2f}")
    base_input["AM Transparency"] = am_transparency

    # Predict PM Transparency
    pm_transparency = predict_pm_transparency(base_input)
    print(f"\n🌇 Predicted PM Transparency: {pm_transparency:.2f}")
    base_input["PM Transparency"] = pm_transparency

    # Predict Fish Survival
    survival_rate = predict_fish_survival(base_input)
    print(f"\n🐟 Predicted Fish Survival Rate: {survival_rate:.2f} %")

In [ ]:
if __name__ == "__main__":
    chained_prediction()
